In [ ]:
''' KPIS:
1) Completion Rate test Vs control 
completed by id / total started by id
2) average time of iteration test vs control 
average (total time by user id/ total steps by user id)
3) Error Rate = number of errors / number of sessions or attempts
4) Step Completion Rate = clients who reached next step / clients who reached current step
5) Average number of step visits per client
average(total steps by client_id)
6)  Median Time per Step test vs control
    Average Time per Step
    Total Completion Time
    Higher completion rate + lower or similar time spent
7)Backtracking Rate = clients who move backwards / clients who started
8)Drop-off Rate = clients who abandon at step / clients who reached that step

9) Client Segment Performance:
Age group
Gender
Tenure
Balance group
Number of accounts
Calls/logons activity '''

In [4]:
import pandas as pd
import numpy as np

# =========================
# 1. Completion Rate KPI
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()

kpi_df["started"] = kpi_df["process_step"].eq("start")
kpi_df["completed"] = kpi_df["process_step"].eq("confirm")

client_completion = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(started=("started", "max"), completed=("completed", "max"))
)

client_completion["started_and_completed"] = client_completion["started"] & client_completion["completed"]

completion_rate_kpi = (
    client_completion.groupby("Variation")
    .agg(
        total_started_clients=("started", "sum"),
        completed_clients=("started_and_completed", "sum")
    )
    .reset_index()
)

completion_rate_kpi["completion_rate_%"] = (
    completion_rate_kpi["completed_clients"] / completion_rate_kpi["total_started_clients"] * 100
).round(2)

completion_rate_kpi

,Variation,total_started_clients,completed_clients,completion_rate_%
0,Control,23397,15318,65.47
1,Test,26679,18412,69.01


In [43]:
def remove_outliers_iqr_by_group(df, value_col, group_col="Variation", multiplier=1.5):
    bounds = (
        df.groupby(group_col)[value_col]
        .quantile([0.25, 0.75])
        .unstack()
        .reset_index()
        .rename(columns={0.25: "q1", 0.75: "q3"})
    )

    bounds["iqr"] = bounds["q3"] - bounds["q1"]
    bounds["lower_bound"] = bounds["q1"] - multiplier * bounds["iqr"]
    bounds["upper_bound"] = bounds["q3"] + multiplier * bounds["iqr"]

    df_with_bounds = df.merge(bounds, on=group_col, how="left")

    df_clean = df_with_bounds[
        df_with_bounds[value_col].between(
            df_with_bounds["lower_bound"],
            df_with_bounds["upper_bound"]
        )
    ].copy()

    return df_clean

In [44]:
# =========================
# 2. Average Iteration Time KPI
# Raw + IQR Outlier Treatment
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# =========================
# Visit-level summary
# =========================

visit_summary = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"], as_index=False)
    .agg(
        visit_start=("date_time", "min"),
        visit_end=("date_time", "max"),
        total_steps=("process_step", "count")
    )
)

visit_summary["visit_duration_seconds"] = (
    visit_summary["visit_end"] - visit_summary["visit_start"]
).dt.total_seconds()

# =========================
# Client-level summary
# =========================

client_iteration_time = (
    visit_summary.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        total_time_seconds=("visit_duration_seconds", "sum"),
        total_steps=("total_steps", "sum")
    )
)

client_iteration_time["avg_iteration_time_seconds_by_client"] = (
    client_iteration_time["total_time_seconds"] / client_iteration_time["total_steps"]
)

client_iteration_time = client_iteration_time[
    client_iteration_time["avg_iteration_time_seconds_by_client"].notna()
].copy()

# =========================
# 2A. Raw KPI
# =========================

avg_iteration_time_kpi = (
    client_iteration_time.groupby("Variation")
    .agg(
        clients=("client_id", "nunique"),
        total_steps=("total_steps", "sum"),
        avg_iteration_time_seconds=("avg_iteration_time_seconds_by_client", "mean"),
        median_iteration_time_seconds=("avg_iteration_time_seconds_by_client", "median")
    )
    .reset_index()
)

avg_iteration_time_kpi["avg_iteration_time_seconds"] = avg_iteration_time_kpi["avg_iteration_time_seconds"].round(2)
avg_iteration_time_kpi["median_iteration_time_seconds"] = avg_iteration_time_kpi["median_iteration_time_seconds"].round(2)

# =========================
# 2B. IQR Outlier Treatment
# =========================

client_iteration_time_iqr = remove_outliers_iqr_by_group(
    client_iteration_time,
    value_col="avg_iteration_time_seconds_by_client",
    group_col="Variation",
    multiplier=1.5
)

avg_iteration_time_iqr_kpi = (
    client_iteration_time_iqr.groupby("Variation")
    .agg(
        clients_included_iqr=("client_id", "nunique"),
        total_steps_iqr=("total_steps", "sum"),
        avg_iteration_time_seconds_iqr=("avg_iteration_time_seconds_by_client", "mean"),
        median_iteration_time_seconds_iqr=("avg_iteration_time_seconds_by_client", "median")
    )
    .reset_index()
)

avg_iteration_time_iqr_kpi["avg_iteration_time_seconds_iqr"] = avg_iteration_time_iqr_kpi["avg_iteration_time_seconds_iqr"].round(2)
avg_iteration_time_iqr_kpi["median_iteration_time_seconds_iqr"] = avg_iteration_time_iqr_kpi["median_iteration_time_seconds_iqr"].round(2)

# =========================
# 2C. Raw vs IQR comparison
# =========================

avg_iteration_comparison_kpi = avg_iteration_time_kpi.merge(
    avg_iteration_time_iqr_kpi,
    on="Variation",
    how="left"
)

avg_iteration_comparison_kpi["clients_removed_iqr"] = (
    avg_iteration_comparison_kpi["clients"] - avg_iteration_comparison_kpi["clients_included_iqr"]
)

avg_iteration_comparison_kpi["clients_removed_iqr_%"] = (
    avg_iteration_comparison_kpi["clients_removed_iqr"] / avg_iteration_comparison_kpi["clients"] * 100
).round(2)

avg_iteration_comparison_kpi

,Variation,clients,total_steps,avg_iteration_time_seconds,median_iteration_time_seconds,clients_included_iqr,total_steps_iqr,avg_iteration_time_seconds_iqr,median_iteration_time_seconds_iqr,clients_removed_iqr,clients_removed_iqr_%
0,Control,23532,140536,54.50,39.0,21867,128042,41.82,36.40,1665,7.08
1,Test,26968,176699,56.66,38.6,24868,159169,42.06,35.83,2100,7.79


In [45]:
avg_iteration_comparison_kpi[
    [
        "Variation",
        "clients",
        "clients_included_iqr",
        "clients_removed_iqr",
        "clients_removed_iqr_%",
        "avg_iteration_time_seconds",
        "median_iteration_time_seconds",
        "avg_iteration_time_seconds_iqr",
        "median_iteration_time_seconds_iqr"
    ]
]

,Variation,clients,clients_included_iqr,clients_removed_iqr,clients_removed_iqr_%,avg_iteration_time_seconds,median_iteration_time_seconds,avg_iteration_time_seconds_iqr,median_iteration_time_seconds_iqr
0,Control,23532,21867,1665,7.08,54.50,39.0,41.82,36.40
1,Test,26968,24868,2100,7.79,56.66,38.6,42.06,35.83


After applying the IQR outlier treatment, the average iteration time became almost identical between Control and Test. Control users spent 41.82 seconds per step on average, while Test users spent 42.06 seconds.

The median iteration time was slightly lower for the Test group, suggesting that for the typical user, the new experience did not increase the time required to move through the process.

This is relevant because the Test group achieved a higher completion rate while maintaining a very similar average iteration time.

In [46]:
# =========================
# 3. Error / Backward Movement Rate KPI
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# Define process order
step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

kpi_df["step_order"] = kpi_df["process_step"].map(step_order)

# Sort by user/session/time
kpi_df = kpi_df.sort_values(["Variation", "client_id", "visit_id", "date_time"])

# Previous step within each session
kpi_df["previous_step_order"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["step_order"]
    .shift(1)
)

# Error = user moves backwards in the process
kpi_df["error_event"] = kpi_df["step_order"] < kpi_df["previous_step_order"]

# Session-level summary: one row per client session
session_errors = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"], as_index=False)
    .agg(
        total_steps=("process_step", "count"),
        error_events=("error_event", "sum"),
        started=("process_step", lambda x: (x == "start").any())
    )
)

session_errors["had_error"] = session_errors["error_events"] > 0
session_errors["started_and_had_error"] = session_errors["started"] & session_errors["had_error"]

# Final KPI table
error_rate_kpi = (
    session_errors.groupby("Variation")
    .agg(
        sessions=("visit_id", "count"),
        started_sessions=("started", "sum"),
        error_events=("error_events", "sum"),
        sessions_with_errors=("had_error", "sum"),
        started_sessions_with_errors=("started_and_had_error", "sum")
    )
    .reset_index()
)

error_rate_kpi["error_events_per_session"] = (
    error_rate_kpi["error_events"] / error_rate_kpi["sessions"]
).round(4)

error_rate_kpi["error_events_per_started_session"] = (
    error_rate_kpi["error_events"] / error_rate_kpi["started_sessions"]
).round(4)

error_rate_kpi["sessions_with_error_rate_%"] = (
    error_rate_kpi["sessions_with_errors"] / error_rate_kpi["sessions"] * 100
).round(2)

error_rate_kpi["started_sessions_with_error_rate_%"] = (
    error_rate_kpi["started_sessions_with_errors"] / error_rate_kpi["started_sessions"] * 100
).round(2)

error_rate_kpi

,Variation,sessions,started_sessions,error_events,sessions_with_errors,started_sessions_with_errors,error_events_per_session,error_events_per_started_session,sessions_with_error_rate_%,started_sessions_with_error_rate_%
0,Control,32243,30962,9581,6522,6430,0.2971,0.3094,20.23,20.77
1,Test,37204,33219,16232,9969,9895,0.4363,0.4886,26.80,29.79


In [47]:
error_rate_kpi_clean = error_rate_kpi[
    [
        "Variation",
        "sessions",
        "started_sessions",
        "error_events",
        "sessions_with_errors",
        "started_sessions_with_errors",
        "error_events_per_session",
        "error_events_per_started_session",
        "sessions_with_error_rate_%",
        "started_sessions_with_error_rate_%"
    ]
]

error_rate_kpi_clean

,Variation,sessions,started_sessions,error_events,sessions_with_errors,started_sessions_with_errors,error_events_per_session,error_events_per_started_session,sessions_with_error_rate_%,started_sessions_with_error_rate_%
0,Control,32243,30962,9581,6522,6430,0.2971,0.3094,20.23,20.77
1,Test,37204,33219,16232,9969,9895,0.4363,0.4886,26.80,29.79


The Error Rate was calculated using backward movements as a proxy for errors. A backward movement occurs when a user moves from a later process step to an earlier one within the same session.

The main KPI used is the percentage of started sessions with at least one backward movement. This measures how often users who began the process experienced friction or confusion during their journey.

In [12]:
# =========================
# 4. Step Completion Rate KPI
# Table ordered by step, then Variation
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()

step_order = ["start", "step_1", "step_2", "step_3", "confirm"]
variation_order = ["Control", "Test"]

# Client-level step flags: one row per client and Variation
client_steps = (
    kpi_df.groupby(["Variation", "client_id"])["process_step"]
    .apply(set)
    .reset_index(name="steps_reached")
)

for step in step_order:
    client_steps[f"reached_{step}"] = client_steps["steps_reached"].apply(lambda x: step in x)

# Calculate step-to-step completion rates
step_completion_results = []

for current_step, next_step in zip(step_order[:-1], step_order[1:]):
    for variation in variation_order:
        temp = client_steps[client_steps["Variation"] == variation]

        clients_current_step = temp[f"reached_{current_step}"].sum()
        clients_next_step = temp[f"reached_{next_step}"].sum()

        step_completion_rate = clients_next_step / clients_current_step if clients_current_step > 0 else np.nan

        step_completion_results.append({
            "Variation": variation,
            "current_step": current_step,
            "next_step": next_step,
            "clients_reached_current_step": clients_current_step,
            "clients_reached_next_step": clients_next_step,
            "step_completion_rate_%": round(step_completion_rate * 100, 2),
            "drop_off_clients": clients_current_step - clients_next_step,
            "drop_off_rate_%": round((1 - step_completion_rate) * 100, 2)
        })

step_completion_kpi = pd.DataFrame(step_completion_results)

step_completion_kpi

,Variation,current_step,next_step,clients_reached_current_step,clients_reached_next_step,step_completion_rate_%,drop_off_clients,drop_off_rate_%
0,Control,start,step_1,23397,20152,86.13,3245,13.87
1,Test,start,step_1,26679,24267,90.96,2412,9.04
2,Control,step_1,step_2,20152,18650,92.55,1502,7.45
3,Test,step_1,step_2,24267,22258,91.72,2009,8.28
4,Control,step_2,step_3,18650,17422,93.42,1228,6.58
5,Test,step_2,step_3,22258,20881,93.81,1377,6.19
6,Control,step_3,confirm,17422,15434,88.59,1988,11.41
7,Test,step_3,confirm,20881,18687,89.49,2194,10.51


In [48]:
# =========================
# 5. Average Number of Step Visits per Client
# Raw + IQR Outlier Treatment
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()

# =========================
# Client-level summary
# =========================

client_step_visits = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        step_visits=("process_step", "count"),
        unique_steps_reached=("process_step", "nunique"),
        sessions=("visit_id", "nunique")
    )
)

# =========================
# 5A. Raw KPI
# =========================

avg_step_visits_kpi = (
    client_step_visits.groupby("Variation")
    .agg(
        clients=("client_id", "nunique"),
        total_step_visits=("step_visits", "sum"),
        avg_step_visits_per_client=("step_visits", "mean"),
        median_step_visits_per_client=("step_visits", "median"),
        avg_unique_steps_reached=("unique_steps_reached", "mean"),
        avg_sessions_per_client=("sessions", "mean")
    )
    .reset_index()
)

avg_step_visits_kpi["avg_step_visits_per_client"] = avg_step_visits_kpi["avg_step_visits_per_client"].round(2)
avg_step_visits_kpi["median_step_visits_per_client"] = avg_step_visits_kpi["median_step_visits_per_client"].round(2)
avg_step_visits_kpi["avg_unique_steps_reached"] = avg_step_visits_kpi["avg_unique_steps_reached"].round(2)
avg_step_visits_kpi["avg_sessions_per_client"] = avg_step_visits_kpi["avg_sessions_per_client"].round(2)

# =========================
# 5B. IQR Outlier Treatment
# =========================

client_step_visits_iqr = remove_outliers_iqr_by_group(
    client_step_visits,
    value_col="step_visits",
    group_col="Variation",
    multiplier=1.5
)

avg_step_visits_iqr_kpi = (
    client_step_visits_iqr.groupby("Variation")
    .agg(
        clients_included_iqr=("client_id", "nunique"),
        total_step_visits_iqr=("step_visits", "sum"),
        avg_step_visits_per_client_iqr=("step_visits", "mean"),
        median_step_visits_per_client_iqr=("step_visits", "median"),
        avg_unique_steps_reached_iqr=("unique_steps_reached", "mean"),
        avg_sessions_per_client_iqr=("sessions", "mean")
    )
    .reset_index()
)

avg_step_visits_iqr_kpi["avg_step_visits_per_client_iqr"] = avg_step_visits_iqr_kpi["avg_step_visits_per_client_iqr"].round(2)
avg_step_visits_iqr_kpi["median_step_visits_per_client_iqr"] = avg_step_visits_iqr_kpi["median_step_visits_per_client_iqr"].round(2)
avg_step_visits_iqr_kpi["avg_unique_steps_reached_iqr"] = avg_step_visits_iqr_kpi["avg_unique_steps_reached_iqr"].round(2)
avg_step_visits_iqr_kpi["avg_sessions_per_client_iqr"] = avg_step_visits_iqr_kpi["avg_sessions_per_client_iqr"].round(2)

# =========================
# 5C. Raw vs IQR comparison
# =========================

avg_step_visits_comparison_kpi = avg_step_visits_kpi.merge(
    avg_step_visits_iqr_kpi,
    on="Variation",
    how="left"
)

avg_step_visits_comparison_kpi["clients_removed_iqr"] = (
    avg_step_visits_comparison_kpi["clients"] - avg_step_visits_comparison_kpi["clients_included_iqr"]
)

avg_step_visits_comparison_kpi["clients_removed_iqr_%"] = (
    avg_step_visits_comparison_kpi["clients_removed_iqr"] / avg_step_visits_comparison_kpi["clients"] * 100
).round(2)

avg_step_visits_comparison_kpi

,Variation,clients,total_step_visits,avg_step_visits_per_client,median_step_visits_per_client,avg_unique_steps_reached,avg_sessions_per_client,clients_included_iqr,total_step_visits_iqr,avg_step_visits_per_client_iqr,median_step_visits_per_client_iqr,avg_unique_steps_reached_iqr,avg_sessions_per_client_iqr,clients_removed_iqr,clients_removed_iqr_%
0,Control,23532,140536,5.97,5.0,4.04,1.37,19214,107402,5.59,5.0,4.33,1.27,4318,18.35
1,Test,26968,176699,6.55,5.0,4.18,1.38,25093,144010,5.74,5.0,4.15,1.28,1875,6.95


In [49]:
avg_step_visits_comparison_kpi_clean = avg_step_visits_comparison_kpi[
    [
        "Variation",
        "clients",
        "clients_included_iqr",
        "clients_removed_iqr",
        "clients_removed_iqr_%",
        "total_step_visits",
        "total_step_visits_iqr",
        "avg_step_visits_per_client",
        "median_step_visits_per_client",
        "avg_step_visits_per_client_iqr",
        "median_step_visits_per_client_iqr",
        "avg_sessions_per_client",
        "avg_sessions_per_client_iqr"
    ]
]

avg_step_visits_comparison_kpi_clean

,Variation,clients,clients_included_iqr,clients_removed_iqr,clients_removed_iqr_%,total_step_visits,total_step_visits_iqr,avg_step_visits_per_client,median_step_visits_per_client,avg_step_visits_per_client_iqr,median_step_visits_per_client_iqr,avg_sessions_per_client,avg_sessions_per_client_iqr
0,Control,23532,19214,4318,18.35,140536,107402,5.97,5.0,5.59,5.0,1.37,1.27
1,Test,26968,25093,1875,6.95,176699,144010,6.55,5.0,5.74,5.0,1.38,1.28


The Average Number of Step Visits per Client measures how many process steps each client visited on average, including repeated visits to the same step.

Because this metric can be affected by clients with unusually high numbers of step visits, an IQR-based outlier treatment was applied separately to the Control and Test groups. Values outside Q1 - 1.5 × IQR and Q3 + 1.5 × IQR were removed.

The raw KPI is shown together with the IQR-adjusted version to compare the original behaviour against a version less affected by extreme navigation patterns.

In [50]:
# =========================
# 6. Time Efficiency KPIs
# Raw + IQR Outlier Treatment
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

step_order = ["start", "step_1", "step_2", "step_3", "confirm"]
variation_order = ["Control", "Test"]

kpi_df["process_step"] = pd.Categorical(kpi_df["process_step"], categories=step_order, ordered=True)

# Sort events chronologically within each session
kpi_df = kpi_df.sort_values(["Variation", "client_id", "visit_id", "date_time"])

In [51]:
# =========================
# 6A. Time per Step
# =========================

kpi_df["next_date_time"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["date_time"]
    .shift(-1)
)

kpi_df["next_process_step"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["process_step"]
    .shift(-1)
)

kpi_df["time_to_next_step_seconds"] = (
    kpi_df["next_date_time"] - kpi_df["date_time"]
).dt.total_seconds()

# Keep valid transitions only
step_time_df = kpi_df[
    (kpi_df["time_to_next_step_seconds"].notna()) &
    (kpi_df["time_to_next_step_seconds"] >= 0) &
    (kpi_df["process_step"] != "confirm")
].copy()

# =========================
# 6A.1 Raw Time per Step KPI
# =========================

time_per_step_kpi = (
    step_time_df.groupby(["process_step", "Variation"], observed=True)
    .agg(
        step_transitions=("time_to_next_step_seconds", "count"),
        avg_time_per_step_seconds=("time_to_next_step_seconds", "mean"),
        median_time_per_step_seconds=("time_to_next_step_seconds", "median")
    )
    .reset_index()
)

time_per_step_kpi["avg_time_per_step_seconds"] = time_per_step_kpi["avg_time_per_step_seconds"].round(2)
time_per_step_kpi["median_time_per_step_seconds"] = time_per_step_kpi["median_time_per_step_seconds"].round(2)

# =========================
# 6A.2 IQR Time per Step KPI
# IQR applied by Variation + process_step
# =========================

step_time_df["iqr_group"] = (
    step_time_df["Variation"].astype(str) + "_" + step_time_df["process_step"].astype(str)
)

step_time_df_iqr = remove_outliers_iqr_by_group(
    step_time_df,
    value_col="time_to_next_step_seconds",
    group_col="iqr_group",
    multiplier=1.5
)

time_per_step_iqr_kpi = (
    step_time_df_iqr.groupby(["process_step", "Variation"], observed=True)
    .agg(
        step_transitions_iqr=("time_to_next_step_seconds", "count"),
        avg_time_per_step_seconds_iqr=("time_to_next_step_seconds", "mean"),
        median_time_per_step_seconds_iqr=("time_to_next_step_seconds", "median")
    )
    .reset_index()
)

time_per_step_iqr_kpi["avg_time_per_step_seconds_iqr"] = time_per_step_iqr_kpi["avg_time_per_step_seconds_iqr"].round(2)
time_per_step_iqr_kpi["median_time_per_step_seconds_iqr"] = time_per_step_iqr_kpi["median_time_per_step_seconds_iqr"].round(2)

# =========================
# 6A.3 Raw vs IQR Time per Step Comparison
# =========================

time_per_step_comparison_kpi = time_per_step_kpi.merge(
    time_per_step_iqr_kpi,
    on=["process_step", "Variation"],
    how="left"
)

time_per_step_comparison_kpi["transitions_removed_iqr"] = (
    time_per_step_comparison_kpi["step_transitions"] - time_per_step_comparison_kpi["step_transitions_iqr"]
)

time_per_step_comparison_kpi["transitions_removed_iqr_%"] = (
    time_per_step_comparison_kpi["transitions_removed_iqr"] / time_per_step_comparison_kpi["step_transitions"] * 100
).round(2)

time_per_step_comparison_kpi["Variation"] = pd.Categorical(
    time_per_step_comparison_kpi["Variation"],
    categories=variation_order,
    ordered=True
)

time_per_step_comparison_kpi["process_step"] = pd.Categorical(
    time_per_step_comparison_kpi["process_step"],
    categories=step_order[:-1],
    ordered=True
)

time_per_step_comparison_kpi = (
    time_per_step_comparison_kpi
    .sort_values(["process_step", "Variation"])
    .reset_index(drop=True)
)

time_per_step_comparison_kpi

,process_step,Variation,step_transitions,avg_time_per_step_seconds,median_time_per_step_seconds,step_transitions_iqr,avg_time_per_step_seconds_iqr,median_time_per_step_seconds_iqr,transitions_removed_iqr,transitions_removed_iqr_%
0,start,Control,35737,66.83,20.0,31610,25.31,17.0,4127,11.55
1,start,Test,46323,61.47,14.0,40347,20.64,12.0,5976,12.90
2,step_1,Control,26043,50.47,20.0,23714,26.06,17.0,2329,8.94
3,step_1,Test,35527,60.68,27.0,31887,31.58,24.0,3640,10.25
4,step_2,Control,24313,92.00,64.0,22838,70.69,60.0,1475,6.07
5,step_2,Test,29573,88.85,61.0,27515,65.32,57.0,2058,6.96
6,step_3,Control,20280,137.14,72.0,18206,83.98,64.0,2074,10.23
7,step_3,Test,23983,129.64,58.0,21334,70.64,50.0,2649,11.05


In [52]:
# =========================
# 6B. Total Completion Time
# =========================

session_step_times = (
    kpi_df.groupby(["Variation", "client_id", "visit_id", "process_step"], observed=True)["date_time"]
    .min()
    .unstack()
    .reset_index()
)

completed_sessions_time = session_step_times[
    session_step_times["start"].notna() &
    session_step_times["confirm"].notna()
].copy()

completed_sessions_time["completion_time_seconds"] = (
    completed_sessions_time["confirm"] - completed_sessions_time["start"]
).dt.total_seconds()

completed_sessions_time = completed_sessions_time[
    completed_sessions_time["completion_time_seconds"] >= 0
].copy()

completed_sessions_time["completion_time_minutes"] = (
    completed_sessions_time["completion_time_seconds"] / 60
)

# =========================
# 6B.1 Raw Completion Time KPI
# =========================

total_completion_time_kpi = (
    completed_sessions_time.groupby("Variation")
    .agg(
        completed_sessions=("visit_id", "count"),
        avg_completion_time_seconds=("completion_time_seconds", "mean"),
        median_completion_time_seconds=("completion_time_seconds", "median"),
        avg_completion_time_minutes=("completion_time_minutes", "mean"),
        median_completion_time_minutes=("completion_time_minutes", "median")
    )
    .reset_index()
)

total_completion_time_kpi["avg_completion_time_seconds"] = total_completion_time_kpi["avg_completion_time_seconds"].round(2)
total_completion_time_kpi["median_completion_time_seconds"] = total_completion_time_kpi["median_completion_time_seconds"].round(2)
total_completion_time_kpi["avg_completion_time_minutes"] = total_completion_time_kpi["avg_completion_time_minutes"].round(2)
total_completion_time_kpi["median_completion_time_minutes"] = total_completion_time_kpi["median_completion_time_minutes"].round(2)

# =========================
# 6B.2 IQR Completion Time KPI
# IQR applied by Variation
# =========================

completed_sessions_time_iqr = remove_outliers_iqr_by_group(
    completed_sessions_time,
    value_col="completion_time_seconds",
    group_col="Variation",
    multiplier=1.5
)

completed_sessions_time_iqr["completion_time_minutes"] = (
    completed_sessions_time_iqr["completion_time_seconds"] / 60
)

total_completion_time_iqr_kpi = (
    completed_sessions_time_iqr.groupby("Variation")
    .agg(
        completed_sessions_included_iqr=("visit_id", "count"),
        avg_completion_time_seconds_iqr=("completion_time_seconds", "mean"),
        median_completion_time_seconds_iqr=("completion_time_seconds", "median"),
        avg_completion_time_minutes_iqr=("completion_time_minutes", "mean"),
        median_completion_time_minutes_iqr=("completion_time_minutes", "median")
    )
    .reset_index()
)

total_completion_time_iqr_kpi["avg_completion_time_seconds_iqr"] = total_completion_time_iqr_kpi["avg_completion_time_seconds_iqr"].round(2)
total_completion_time_iqr_kpi["median_completion_time_seconds_iqr"] = total_completion_time_iqr_kpi["median_completion_time_seconds_iqr"].round(2)
total_completion_time_iqr_kpi["avg_completion_time_minutes_iqr"] = total_completion_time_iqr_kpi["avg_completion_time_minutes_iqr"].round(2)
total_completion_time_iqr_kpi["median_completion_time_minutes_iqr"] = total_completion_time_iqr_kpi["median_completion_time_minutes_iqr"].round(2)

# =========================
# 6B.3 Raw vs IQR Completion Time Comparison
# =========================

completion_time_comparison_kpi = total_completion_time_kpi.merge(
    total_completion_time_iqr_kpi,
    on="Variation",
    how="left"
)

completion_time_comparison_kpi["completed_sessions_removed_iqr"] = (
    completion_time_comparison_kpi["completed_sessions"] - completion_time_comparison_kpi["completed_sessions_included_iqr"]
)

completion_time_comparison_kpi["completed_sessions_removed_iqr_%"] = (
    completion_time_comparison_kpi["completed_sessions_removed_iqr"] / completion_time_comparison_kpi["completed_sessions"] * 100
).round(2)

completion_time_comparison_kpi

,Variation,completed_sessions,avg_completion_time_seconds,median_completion_time_seconds,avg_completion_time_minutes,median_completion_time_minutes,completed_sessions_included_iqr,avg_completion_time_seconds_iqr,median_completion_time_seconds_iqr,avg_completion_time_minutes_iqr,median_completion_time_minutes_iqr,completed_sessions_removed_iqr,completed_sessions_removed_iqr_%
0,Control,14925,391.10,271.0,6.52,4.52,13727,301.71,253.0,5.03,4.22,1198,8.03
1,Test,17934,374.64,237.0,6.24,3.95,16408,271.07,219.0,4.52,3.65,1526,8.51


In [53]:
# =========================
# 6C. Final Completion Efficiency Summary
# =========================

completion_efficiency_kpi = completion_rate_kpi.merge(
    completion_time_comparison_kpi,
    on="Variation",
    how="left"
)

completion_efficiency_kpi = completion_efficiency_kpi[
    [
        "Variation",
        "total_started_clients",
        "completed_clients",
        "completion_rate_%",
        "completed_sessions",
        "completed_sessions_included_iqr",
        "completed_sessions_removed_iqr",
        "completed_sessions_removed_iqr_%",
        "avg_completion_time_minutes",
        "median_completion_time_minutes",
        "avg_completion_time_minutes_iqr",
        "median_completion_time_minutes_iqr"
    ]
]

completion_efficiency_kpi

,Variation,total_started_clients,completed_clients,completion_rate_%,completed_sessions,completed_sessions_included_iqr,completed_sessions_removed_iqr,completed_sessions_removed_iqr_%,avg_completion_time_minutes,median_completion_time_minutes,avg_completion_time_minutes_iqr,median_completion_time_minutes_iqr
0,Control,23397,15318,65.47,14925,13727,1198,8.03,6.52,4.52,5.03,4.22
1,Test,26679,18412,69.01,17934,16408,1526,8.51,6.24,3.95,4.52,3.65


Time Efficiency was analysed using both step-level and full completion-time metrics.

First, Time per Step measured the time between one process step and the next. Because each step may naturally require a different amount of time, IQR-based outlier treatment was applied separately by Variation and process step.

Second, Total Completion Time measured the time between start and confirm for completed sessions only. For this metric, IQR-based outlier treatment was applied separately by Variation.

The final efficiency comparison combines completion rate with completion time. This allows us to evaluate whether the Test experience improved completion without meaningfully increasing the time required to complete the process.

In [54]:
# =========================
# 7. Backtracking Rate KPI
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# Define process order
step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

kpi_df["step_order"] = kpi_df["process_step"].map(step_order)

# Sort events chronologically within each client/session
kpi_df = kpi_df.sort_values(["Variation", "client_id", "visit_id", "date_time"])

# Previous step within the same session
kpi_df["previous_step_order"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["step_order"]
    .shift(1)
)

# Backtracking event = current step is earlier than previous step
kpi_df["backtracking_event"] = kpi_df["step_order"] < kpi_df["previous_step_order"]

# Client-level summary
client_backtracking = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        started=("process_step", lambda x: (x == "start").any()),
        backtracked=("backtracking_event", "max"),
        backtracking_events=("backtracking_event", "sum")
    )
)

# Only count backtracking among clients who started
client_backtracking["started_and_backtracked"] = (
    client_backtracking["started"] & client_backtracking["backtracked"]
)

# Final KPI table
backtracking_rate_kpi = (
    client_backtracking.groupby("Variation")
    .agg(
        total_clients=("client_id", "nunique"),
        started_clients=("started", "sum"),
        clients_who_backtracked=("started_and_backtracked", "sum"),
        total_backtracking_events=("backtracking_events", "sum")
    )
    .reset_index()
)

backtracking_rate_kpi["backtracking_rate_%"] = (
    backtracking_rate_kpi["clients_who_backtracked"] / backtracking_rate_kpi["started_clients"] * 100
).round(2)

backtracking_rate_kpi["backtracking_events_per_started_client"] = (
    backtracking_rate_kpi["total_backtracking_events"] / backtracking_rate_kpi["started_clients"]
).round(4)

backtracking_rate_kpi["backtracking_events_per_backtracked_client"] = (
    backtracking_rate_kpi["total_backtracking_events"] / backtracking_rate_kpi["clients_who_backtracked"]
).round(2)

backtracking_rate_kpi

,Variation,total_clients,started_clients,clients_who_backtracked,total_backtracking_events,backtracking_rate_%,backtracking_events_per_started_client,backtracking_events_per_backtracked_client
0,Control,23532,23397,6116,9581,26.14,0.4095,1.57
1,Test,26968,26679,8995,16232,33.72,0.6084,1.80


The Backtracking Rate measures the percentage of started clients who moved backwards at least once during the digital process.

A backward movement was defined as moving from a later step to an earlier step within the same session, such as from step_3 back to step_2. This behaviour may indicate friction, confusion, hesitation, or difficulty progressing through the process.

The KPI was calculated at client level, meaning each client is counted only once in the backtracking rate, even if they moved backwards multiple times.

In [55]:
# =========================
# Client Segment Performance
# Completion Rate + IQR-adjusted behaviour KPIs
# =========================

demo_df = pd.read_csv("1.1_Clean_Files/df1_demo_clean_plus_experiments_clients.csv")

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# =========================
# 1. Client-level behaviour KPIs
# =========================

client_completion = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        started=("process_step", lambda x: (x == "start").any()),
        completed=("process_step", lambda x: (x == "confirm").any())
    )
)

client_completion["started_and_completed"] = client_completion["started"] & client_completion["completed"]

visit_summary = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"], as_index=False)
    .agg(
        visit_start=("date_time", "min"),
        visit_end=("date_time", "max"),
        step_visits=("process_step", "count")
    )
)

visit_summary["visit_duration_seconds"] = (
    visit_summary["visit_end"] - visit_summary["visit_start"]
).dt.total_seconds()

client_iteration = (
    visit_summary.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        total_time_seconds=("visit_duration_seconds", "sum"),
        total_step_visits=("step_visits", "sum")
    )
)

client_iteration["avg_iteration_time_seconds"] = (
    client_iteration["total_time_seconds"] / client_iteration["total_step_visits"]
)

client_step_visits = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        step_visits_per_client=("process_step", "count"),
        sessions=("visit_id", "nunique")
    )
)

client_behavior = (
    client_completion
    .merge(client_iteration[["Variation", "client_id", "avg_iteration_time_seconds"]], on=["Variation", "client_id"], how="left")
    .merge(client_step_visits[["Variation", "client_id", "step_visits_per_client", "sessions"]], on=["Variation", "client_id"], how="left")
)

In [56]:
# =========================
# 2. IQR cleaning helper
# Keeps clients, but sets outlier metric values to NaN
# =========================

def add_iqr_clean_metric(df, value_col, clean_col, group_col="Variation", multiplier=1.5):
    bounds = (
        df.groupby(group_col)[value_col]
        .quantile([0.25, 0.75])
        .unstack()
        .reset_index()
        .rename(columns={0.25: "q1", 0.75: "q3"})
    )

    bounds["iqr"] = bounds["q3"] - bounds["q1"]
    bounds["lower_bound"] = bounds["q1"] - multiplier * bounds["iqr"]
    bounds["upper_bound"] = bounds["q3"] + multiplier * bounds["iqr"]

    df_clean = df.merge(bounds, on=group_col, how="left")

    df_clean[clean_col] = np.where(
        df_clean[value_col].between(df_clean["lower_bound"], df_clean["upper_bound"]),
        df_clean[value_col],
        np.nan
    )

    df_clean = df_clean.drop(columns=["q1", "q3", "iqr", "lower_bound", "upper_bound"])

    return df_clean

In [57]:
# =========================
# 3. Apply IQR treatment to continuous behaviour KPIs
# =========================

client_behavior = add_iqr_clean_metric(
    client_behavior,
    value_col="avg_iteration_time_seconds",
    clean_col="avg_iteration_time_seconds_iqr",
    group_col="Variation",
    multiplier=1.5
)

client_behavior = add_iqr_clean_metric(
    client_behavior,
    value_col="step_visits_per_client",
    clean_col="step_visits_per_client_iqr",
    group_col="Variation",
    multiplier=1.5
)

client_behavior.head()

,Variation,client_id,started,completed,started_and_completed,avg_iteration_time_seconds,step_visits_per_client,sessions,avg_iteration_time_seconds_iqr,step_visits_per_client_iqr
0,Control,1028,True,False,False,59.777778,9,1,59.777778,9.0
1,Control,1104,True,False,False,0.000000,2,2,0.000000,2.0
2,Control,1186,True,False,False,5.500000,4,2,5.500000,4.0
3,Control,1195,True,True,True,49.000000,5,1,49.000000,5.0
4,Control,1197,True,True,True,13.571429,7,1,13.571429,7.0


In [58]:
# =========================
# 4. Client Profile Segments
# =========================

demo_base = demo_df[demo_df["Variation"].isin(["Test", "Control"])].copy()

demo_base["gender_clean"] = (
    demo_base["gender"]
    .astype(str)
    .str.strip()
    .replace({
        "Male": "M",
        "Female": "F",
        "male": "M",
        "female": "F",
        "M": "M",
        "F": "F"
    })
)

client_profile = (
    demo_base.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        client_age=("client_age", "median"),
        gender=("gender_clean", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        client_tenure_years=("client_tenure_years", "median"),
        balance=("balance", "median")
    )
)

client_profile = client_profile[client_profile["gender"].isin(["M", "F"])].copy()

# Age groups in bins of 10 years
age_min = int(np.floor(client_profile["client_age"].min() / 10) * 10)
age_max = int(np.ceil(client_profile["client_age"].max() / 10) * 10)

age_bins = list(range(age_min, age_max + 10, 10))
age_labels = [f"{age_bins[i]}-{age_bins[i+1]-1}" for i in range(len(age_bins)-1)]

client_profile["age_group"] = pd.cut(client_profile["client_age"], bins=age_bins, labels=age_labels, right=False, include_lowest=True)

# Tenure groups
client_profile["tenure_group"] = pd.cut(
    client_profile["client_tenure_years"],
    bins=[0, 10, 20, np.inf],
    labels=["0-10 years", "10-20 years", "20+ years"],
    right=False,
    include_lowest=True
)

# Balance groups
client_profile["balance_group"] = pd.cut(
    client_profile["balance"],
    bins=[-np.inf, 100000, 200000, np.inf],
    labels=["Under 100K", "100K-200K", "Over 200K"],
    right=False
)

client_profile.head()

,Variation,client_id,client_age,gender,client_tenure_years,balance,age_group,tenure_group,balance_group
0,Control,1195,56.5,M,9.0,43738.68,50-59,0-10 years,Under 100K
1,Control,1197,35.5,M,20.0,115749.34,30-39,20+ years,100K-200K
2,Control,3647,24.5,F,13.0,46313.66,20-29,10-20 years,Under 100K
5,Control,8101,62.5,F,10.0,35340.59,60-69,10-20 years,Under 100K
6,Control,13831,55.5,M,19.0,60679.55,50-59,10-20 years,Under 100K


In [59]:
# =========================
# 5. Merge Behaviour + Segments
# =========================

client_segment_performance = client_behavior.merge(
    client_profile,
    on=["Variation", "client_id"],
    how="inner"
)

client_segment_performance.head()

,Variation,client_id,started,completed,started_and_completed,avg_iteration_time_seconds,step_visits_per_client,sessions,avg_iteration_time_seconds_iqr,step_visits_per_client_iqr,client_age,gender,client_tenure_years,balance,age_group,tenure_group,balance_group
0,Control,1195,True,True,True,49.000000,5,1,49.000000,5.0,56.5,M,9.0,43738.68,50-59,0-10 years,Under 100K
1,Control,1197,True,True,True,13.571429,7,1,13.571429,7.0,35.5,M,20.0,115749.34,30-39,20+ years,100K-200K
2,Control,3647,True,False,False,134.000000,6,2,134.000000,6.0,24.5,F,13.0,46313.66,20-29,10-20 years,Under 100K
3,Control,8101,True,True,True,190.800000,5,1,NaN,5.0,62.5,F,10.0,35340.59,60-69,10-20 years,Under 100K
4,Control,13831,True,False,False,266.444444,9,2,NaN,9.0,55.5,M,19.0,60679.55,50-59,10-20 years,Under 100K


In [60]:
# =========================
# 6. Segment Performance Function
# =========================

def calculate_segment_performance(df, segment_col):
    segment_kpi = (
        df.groupby([segment_col, "Variation"], observed=True)
        .agg(
            clients=("client_id", "nunique"),
            started_clients=("started", "sum"),
            completed_clients=("started_and_completed", "sum"),

            avg_iteration_time_seconds_raw=("avg_iteration_time_seconds", "mean"),
            median_iteration_time_seconds_raw=("avg_iteration_time_seconds", "median"),
            avg_iteration_time_seconds_iqr=("avg_iteration_time_seconds_iqr", "mean"),
            median_iteration_time_seconds_iqr=("avg_iteration_time_seconds_iqr", "median"),
            clients_included_iteration_iqr=("avg_iteration_time_seconds_iqr", "count"),

            avg_step_visits_per_client_raw=("step_visits_per_client", "mean"),
            median_step_visits_per_client_raw=("step_visits_per_client", "median"),
            avg_step_visits_per_client_iqr=("step_visits_per_client_iqr", "mean"),
            median_step_visits_per_client_iqr=("step_visits_per_client_iqr", "median"),
            clients_included_steps_iqr=("step_visits_per_client_iqr", "count")
        )
        .reset_index()
    )

    segment_kpi["completion_rate_%"] = (
        segment_kpi["completed_clients"] / segment_kpi["started_clients"] * 100
    ).round(2)

    cols_to_round = [
        "avg_iteration_time_seconds_raw",
        "median_iteration_time_seconds_raw",
        "avg_iteration_time_seconds_iqr",
        "median_iteration_time_seconds_iqr",
        "avg_step_visits_per_client_raw",
        "median_step_visits_per_client_raw",
        "avg_step_visits_per_client_iqr",
        "median_step_visits_per_client_iqr"
    ]

    for col in cols_to_round:
        segment_kpi[col] = segment_kpi[col].round(2)

    segment_kpi["Variation"] = pd.Categorical(segment_kpi["Variation"], categories=["Control", "Test"], ordered=True)

    segment_kpi = segment_kpi.sort_values([segment_col, "Variation"]).reset_index(drop=True)

    return segment_kpi

In [61]:
# =========================
# 7. Segment KPI Tables
# =========================

age_group_performance = calculate_segment_performance(client_segment_performance, "age_group")
gender_performance = calculate_segment_performance(client_segment_performance, "gender")
tenure_group_performance = calculate_segment_performance(client_segment_performance, "tenure_group")
balance_group_performance = calculate_segment_performance(client_segment_performance, "balance_group")

In [62]:
age_group_performance

,age_group,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds_raw,median_iteration_time_seconds_raw,avg_iteration_time_seconds_iqr,median_iteration_time_seconds_iqr,clients_included_iteration_iqr,avg_step_visits_per_client_raw,median_step_visits_per_client_raw,avg_step_visits_per_client_iqr,median_step_visits_per_client_iqr,clients_included_steps_iqr,completion_rate_%
0,10-19,Control,5,5,1,63.01,0.00,6.41,0.00,4,3.60,1.0,7.50,7.5,2,20.00
1,10-19,Test,3,3,0,0.00,0.00,0.00,0.00,3,1.00,1.0,1.00,1.0,3,0.00
2,20-29,Control,149,149,83,43.11,31.60,32.89,30.02,142,4.46,5.0,5.01,5.0,109,55.70
3,20-29,Test,161,159,93,37.16,26.29,31.03,24.42,154,4.60,5.0,4.51,5.0,160,58.49
4,30-39,Control,699,694,475,57.12,42.17,43.31,38.93,646,5.70,5.0,5.53,5.0,598,68.44
5,30-39,Test,908,892,629,56.73,38.16,41.52,35.40,835,6.29,5.0,5.82,5.0,871,70.52
6,40-49,Control,1299,1296,927,57.34,43.22,46.92,40.54,1214,6.81,5.0,5.68,5.0,1071,71.53
7,40-49,Test,1556,1545,1142,61.62,42.90,46.82,39.50,1424,7.44,6.0,6.17,6.0,1389,73.92
8,50-59,Control,1229,1224,908,56.48,41.80,44.66,39.00,1136,6.33,5.0,5.79,5.0,1057,74.18
9,50-59,Test,1586,1573,1151,59.32,41.82,45.00,38.80,1457,6.89,6.0,6.00,5.0,1465,73.17


In [63]:
gender_performance

,gender,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds_raw,median_iteration_time_seconds_raw,avg_iteration_time_seconds_iqr,median_iteration_time_seconds_iqr,clients_included_iteration_iqr,avg_step_visits_per_client_raw,median_step_visits_per_client_raw,avg_step_visits_per_client_iqr,median_step_visits_per_client_iqr,clients_included_steps_iqr,completion_rate_%
0,F,Control,2077,2069,1437,56.22,41.20,43.15,38.60,1922,6.12,5.0,5.56,5.0,1755,69.45
1,F,Test,2568,2543,1774,58.11,39.89,43.36,37.14,2355,6.63,5.0,5.79,5.0,2382,69.76
2,M,Control,1702,1696,1173,55.65,40.79,44.56,38.57,1587,6.19,5.0,5.69,5.0,1377,69.16
3,M,Test,2054,2030,1498,58.09,40.33,44.41,37.80,1897,6.84,5.0,5.97,5.0,1908,73.79


In [64]:
tenure_group_performance

,tenure_group,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds_raw,median_iteration_time_seconds_raw,avg_iteration_time_seconds_iqr,median_iteration_time_seconds_iqr,clients_included_iteration_iqr,avg_step_visits_per_client_raw,median_step_visits_per_client_raw,avg_step_visits_per_client_iqr,median_step_visits_per_client_iqr,clients_included_steps_iqr,completion_rate_%
0,0-10 years,Control,971,967,687,55.02,40.80,43.55,39.00,901,6.04,5.0,5.58,5.0,837,71.04
1,0-10 years,Test,1230,1213,861,58.37,38.60,43.19,35.79,1128,6.54,5.0,5.76,5.0,1146,70.98
2,10-20 years,Control,2396,2387,1663,57.12,42.15,44.64,39.00,2222,6.39,5.0,5.69,5.0,1947,69.67
3,10-20 years,Test,2898,2870,2079,59.07,41.00,44.60,38.14,2662,6.98,6.0,6.01,5.0,2665,72.44
4,20+ years,Control,412,411,260,51.49,35.88,39.44,34.42,386,5.01,5.0,5.34,5.0,348,63.26
5,20+ years,Test,494,490,332,51.75,38.44,40.96,34.96,462,5.68,5.0,5.35,5.0,479,67.76


In [65]:
balance_group_performance

,balance_group,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds_raw,median_iteration_time_seconds_raw,avg_iteration_time_seconds_iqr,median_iteration_time_seconds_iqr,clients_included_iteration_iqr,avg_step_visits_per_client_raw,median_step_visits_per_client_raw,avg_step_visits_per_client_iqr,median_step_visits_per_client_iqr,clients_included_steps_iqr,completion_rate_%
0,Under 100K,Control,2764,2756,1997,57.42,42.00,44.86,39.25,2565,6.40,5.0,5.72,5.0,2298,72.46
1,Under 100K,Test,3399,3366,2479,59.19,41.60,45.03,38.68,3118,7.01,6.0,6.03,5.0,3115,73.65
2,100K-200K,Control,723,718,485,55.22,41.50,44.17,38.27,672,5.81,5.0,5.50,5.0,621,67.55
3,100K-200K,Test,947,939,660,58.62,37.60,42.30,34.60,870,6.35,5.0,5.78,5.0,902,70.29
4,Over 200K,Control,292,291,128,44.03,30.50,32.76,29.17,272,4.57,5.0,4.89,5.0,213,43.99
5,Over 200K,Test,276,268,133,42.84,30.65,34.74,29.43,264,4.49,5.0,4.37,5.0,273,49.63


# AB TESTING MDFKRSSSS


H0: completion_rate_Test <= completion_rate_Control

H1: completion_rate_Test > completion_rate_Control

In [66]:
# =========================
# A/B Testing - Completion Rate
# =========================

from statsmodels.stats.proportion import proportions_ztest
import pandas as pd

# Hypotheses:
# H0: completion_rate_Test <= completion_rate_Control
# H1: completion_rate_Test > completion_rate_Control

alpha = 0.05

# Extract counts
control_completed = completion_rate_kpi.loc[completion_rate_kpi["Variation"] == "Control", "completed_clients"].iloc[0]
test_completed = completion_rate_kpi.loc[completion_rate_kpi["Variation"] == "Test", "completed_clients"].iloc[0]

control_started = completion_rate_kpi.loc[completion_rate_kpi["Variation"] == "Control", "total_started_clients"].iloc[0]
test_started = completion_rate_kpi.loc[completion_rate_kpi["Variation"] == "Test", "total_started_clients"].iloc[0]

# Completion rates
control_rate = control_completed / control_started
test_rate = test_completed / test_started

# Counts and observations
count = [test_completed, control_completed]
nobs = [test_started, control_started]

# One-sided two-proportion z-test: Test > Control
z_stat, p_value = proportions_ztest(count=count, nobs=nobs, alternative="larger")

# Lift
absolute_lift_pp = (test_rate - control_rate) * 100
relative_lift_pct = ((test_rate / control_rate) - 1) * 100

# Result interpretation
if p_value < alpha:
    test_result = "Reject H0"
    conclusion = "The Test group has a statistically significant higher completion rate."
else:
    test_result = "Fail to reject H0"
    conclusion = "There is not enough evidence to confirm that Test improved completion rate."

# Summary table
ab_completion_result = pd.DataFrame({
    "metric": ["completion_rate"],
    "control_rate_%": [round(control_rate * 100, 2)],
    "test_rate_%": [round(test_rate * 100, 2)],
    "absolute_lift_pp": [round(absolute_lift_pp, 2)],
    "relative_lift_%": [round(relative_lift_pct, 2)],
    "z_statistic": [round(z_stat, 4)],
    "p_value": [round(p_value, 6)],
    "alpha": [alpha],
    "result": [test_result],
    "conclusion": [conclusion]
})

ab_completion_result

,metric,control_rate_%,test_rate_%,absolute_lift_pp,relative_lift_%,z_statistic,p_value,alpha,result,conclusion
0,completion_rate,65.47,69.01,3.54,5.41,8.4364,0.0,0.05,Reject H0,The Test group has a statistically significant...


In [78]:
print(f"Control completion rate: {control_rate:.2%}")
print(f"Test completion rate: {test_rate:.2%}")
print(f"Absolute lift: {absolute_lift_pp:.2f} percentage points")
print(f"Relative lift: {relative_lift_pct:.2f}%")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:}")
print(f"Result: {test_result}")
print(f"Conclusion: {conclusion}")

Control completion rate: 65.47%
Test completion rate: 69.01%
Absolute lift: 3.54 percentage points
Relative lift: 5.41%
Z-statistic: 8.4364
P-value: 0.00041947810326332794
Result: Reject H0
Conclusion: There is a statistically significant difference in average duration between Test and Control.


A two-proportion z-test was used to evaluate whether the Test group achieved a statistically significant improvement in completion rate compared with the Control group.

The null hypothesis states that the Test completion rate is less than or equal to the Control completion rate. The alternative hypothesis states that the Test completion rate is higher than the Control completion rate.

Using a significance level of 0.05, if the p-value is below 0.05, we reject the null hypothesis and conclude that the Test version produced a statistically significant improvement in completion rate.

This is a one-sided test because the business question is specifically whether the Test version improved completion, not simply whether the two groups are different.

H0: avg_duration_Test >= avg_duration_Control

H1: avg_duration_Test < avg_duration_Control

In [68]:
# =========================
# A/B Testing - Average Iteration Duration
# Raw + IQR
# =========================

from scipy.stats import ttest_ind
import pandas as pd
import numpy as np

# Hypotheses:
# H0: avg_duration_Test >= avg_duration_Control
# H1: avg_duration_Test < avg_duration_Control

alpha = 0.05

# =========================
# 1. Average Iteration Time by Client
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

visit_summary = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"], as_index=False)
    .agg(
        visit_start=("date_time", "min"),
        visit_end=("date_time", "max"),
        total_steps=("process_step", "count")
    )
)

visit_summary["visit_duration_seconds"] = (
    visit_summary["visit_end"] - visit_summary["visit_start"]
).dt.total_seconds()

client_iteration_time = (
    visit_summary.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        total_time_seconds=("visit_duration_seconds", "sum"),
        total_steps=("total_steps", "sum")
    )
)

client_iteration_time["avg_iteration_time_seconds"] = (
    client_iteration_time["total_time_seconds"] / client_iteration_time["total_steps"]
)

client_iteration_time = client_iteration_time[
    client_iteration_time["avg_iteration_time_seconds"].notna()
].copy()

In [69]:
# =========================
# 2. Raw Welch t-test
# =========================

control_duration_raw = client_iteration_time.loc[
    client_iteration_time["Variation"] == "Control",
    "avg_iteration_time_seconds"
]

test_duration_raw = client_iteration_time.loc[
    client_iteration_time["Variation"] == "Test",
    "avg_iteration_time_seconds"
]

t_stat_raw, p_value_raw = ttest_ind(
    test_duration_raw,
    control_duration_raw,
    equal_var=False,
    alternative="less"
)

control_mean_raw = control_duration_raw.mean()
test_mean_raw = test_duration_raw.mean()

raw_result = "Reject H0" if p_value_raw < alpha else "Fail to reject H0"

raw_conclusion = (
    "The Test group has a statistically significant lower average duration."
    if p_value_raw < alpha
    else "There is not enough evidence to confirm that Test reduced average duration."
)

In [70]:
# =========================
# 3. IQR Welch t-test
# =========================

client_iteration_time_iqr = remove_outliers_iqr_by_group(
    client_iteration_time,
    value_col="avg_iteration_time_seconds",
    group_col="Variation",
    multiplier=1.5
)

control_duration_iqr = client_iteration_time_iqr.loc[
    client_iteration_time_iqr["Variation"] == "Control",
    "avg_iteration_time_seconds"
]

test_duration_iqr = client_iteration_time_iqr.loc[
    client_iteration_time_iqr["Variation"] == "Test",
    "avg_iteration_time_seconds"
]

t_stat_iqr, p_value_iqr = ttest_ind(
    test_duration_iqr,
    control_duration_iqr,
    equal_var=False,
    alternative="less"
)

control_mean_iqr = control_duration_iqr.mean()
test_mean_iqr = test_duration_iqr.mean()

iqr_result = "Reject H0" if p_value_iqr < alpha else "Fail to reject H0"

iqr_conclusion = (
    "The Test group has a statistically significant lower average duration after IQR outlier treatment."
    if p_value_iqr < alpha
    else "There is not enough evidence to confirm that Test reduced average duration after IQR outlier treatment."
)

In [71]:
# =========================
# 4. Summary Table
# =========================

ab_duration_result = pd.DataFrame({
    "method": ["Raw", "IQR"],
    "control_mean_seconds": [round(control_mean_raw, 2), round(control_mean_iqr, 2)],
    "test_mean_seconds": [round(test_mean_raw, 2), round(test_mean_iqr, 2)],
    "difference_test_minus_control_seconds": [
        round(test_mean_raw - control_mean_raw, 2),
        round(test_mean_iqr - control_mean_iqr, 2)
    ],
    "t_statistic": [round(t_stat_raw, 4), round(t_stat_iqr, 4)],
    "p_value": [round(p_value_raw, 6), round(p_value_iqr, 6)],
    "alpha": [alpha, alpha],
    "result": [raw_result, iqr_result],
    "conclusion": [raw_conclusion, iqr_conclusion]
})

ab_duration_result

,method,control_mean_seconds,test_mean_seconds,difference_test_minus_control_seconds,t_statistic,p_value,alpha,result,conclusion
0,Raw,54.50,56.66,2.16,3.5278,0.999790,0.05,Fail to reject H0,There is not enough evidence to confirm that T...
1,IQR,41.82,42.06,0.24,0.8533,0.803252,0.05,Fail to reject H0,There is not enough evidence to confirm that T...


A Welch two-sample t-test was used to compare the average iteration duration between the Test and Control groups.

The null hypothesis states that the Test group has an average duration greater than or equal to the Control group. The alternative hypothesis states that the Test group has a lower average duration.

A one-sided test was used because the business question is specifically whether the Test version reduced the average time spent per step.

Because time-based behavioural data is often affected by outliers, the test was performed using both the raw data and an IQR-adjusted version. The IQR method removes values outside Q1 - 1.5 × IQR and Q3 + 1.5 × IQR within each Variation group.

H0: avg_duration_Test = avg_duration_Control

H1: avg_duration_Test != avg_duration_Control

In [72]:
# =========================
# A/B Testing - Average Iteration Duration
# Two-sided Welch t-test
# =========================

from scipy.stats import ttest_ind
import pandas as pd

# Hypotheses:
# H0: avg_duration_Test = avg_duration_Control
# H1: avg_duration_Test != avg_duration_Control

alpha = 0.05

control_duration = client_iteration_time.loc[
    client_iteration_time["Variation"] == "Control",
    "avg_iteration_time_seconds"
]

test_duration = client_iteration_time.loc[
    client_iteration_time["Variation"] == "Test",
    "avg_iteration_time_seconds"
]

# Welch t-test: does not assume equal variances
t_stat, p_value = ttest_ind(
    test_duration,
    control_duration,
    equal_var=False,
    alternative="two-sided"
)

control_mean = control_duration.mean()
test_mean = test_duration.mean()

duration_difference = test_mean - control_mean

if p_value < alpha:
    result = "Reject H0"
    conclusion = "There is a statistically significant difference in average duration between Test and Control."
else:
    result = "Fail to reject H0"
    conclusion = "There is not enough evidence to confirm a statistically significant difference in average duration between Test and Control."

ab_duration_two_sided_result = pd.DataFrame({
    "metric": ["avg_iteration_time_seconds"],
    "control_mean_seconds": [round(control_mean, 2)],
    "test_mean_seconds": [round(test_mean, 2)],
    "difference_test_minus_control_seconds": [round(duration_difference, 2)],
    "t_statistic": [round(t_stat, 4)],
    "p_value": [round(p_value, 6)],
    "alpha": [alpha],
    "result": [result],
    "conclusion": [conclusion]
})

ab_duration_two_sided_result

,metric,control_mean_seconds,test_mean_seconds,difference_test_minus_control_seconds,t_statistic,p_value,alpha,result,conclusion
0,avg_iteration_time_seconds,54.5,56.66,2.16,3.5278,0.000419,0.05,Reject H0,There is a statistically significant differenc...


In [73]:
# =========================
# A/B Testing - Average Iteration Duration
# Two-sided Welch t-test with IQR treatment
# =========================

client_iteration_time_iqr = remove_outliers_iqr_by_group(
    client_iteration_time,
    value_col="avg_iteration_time_seconds",
    group_col="Variation",
    multiplier=1.5
)

control_duration_iqr = client_iteration_time_iqr.loc[
    client_iteration_time_iqr["Variation"] == "Control",
    "avg_iteration_time_seconds"
]

test_duration_iqr = client_iteration_time_iqr.loc[
    client_iteration_time_iqr["Variation"] == "Test",
    "avg_iteration_time_seconds"
]

t_stat_iqr, p_value_iqr = ttest_ind(
    test_duration_iqr,
    control_duration_iqr,
    equal_var=False,
    alternative="two-sided"
)

control_mean_iqr = control_duration_iqr.mean()
test_mean_iqr = test_duration_iqr.mean()

duration_difference_iqr = test_mean_iqr - control_mean_iqr

if p_value_iqr < alpha:
    result_iqr = "Reject H0"
    conclusion_iqr = "There is a statistically significant difference in average duration between Test and Control after IQR outlier treatment."
else:
    result_iqr = "Fail to reject H0"
    conclusion_iqr = "There is not enough evidence to confirm a statistically significant difference in average duration between Test and Control after IQR outlier treatment."

ab_duration_two_sided_iqr_result = pd.DataFrame({
    "method": ["IQR"],
    "metric": ["avg_iteration_time_seconds"],
    "control_mean_seconds": [round(control_mean_iqr, 2)],
    "test_mean_seconds": [round(test_mean_iqr, 2)],
    "difference_test_minus_control_seconds": [round(duration_difference_iqr, 2)],
    "t_statistic": [round(t_stat_iqr, 4)],
    "p_value": [round(p_value_iqr, 6)],
    "alpha": [alpha],
    "result": [result_iqr],
    "conclusion": [conclusion_iqr]
})

ab_duration_two_sided_iqr_result

,method,metric,control_mean_seconds,test_mean_seconds,difference_test_minus_control_seconds,t_statistic,p_value,alpha,result,conclusion
0,IQR,avg_iteration_time_seconds,41.82,42.06,0.24,0.8533,0.393495,0.05,Fail to reject H0,There is not enough evidence to confirm a stat...


The raw duration comparison showed a statistically significant difference between Test and Control, with Test users spending 2.16 seconds more per step on average.

However, after applying IQR-based outlier treatment, the difference dropped to only 0.24 seconds and was no longer statistically significant.

This suggests that the raw difference was likely driven by extreme duration values rather than a meaningful difference in typical user behaviour. Therefore, there is not enough evidence to conclude that the Test experience meaningfully changed average iteration time after accounting for outliers.

In [74]:
ab_duration_combined_result = pd.concat(
    [
        ab_duration_two_sided_result.assign(method="Raw"),
        ab_duration_two_sided_iqr_result
    ],
    ignore_index=True
)

ab_duration_combined_result = ab_duration_combined_result[
    [
        "method",
        "metric",
        "control_mean_seconds",
        "test_mean_seconds",
        "difference_test_minus_control_seconds",
        "t_statistic",
        "p_value",
        "alpha",
        "result",
        "conclusion"
    ]
]

ab_duration_combined_result

,method,metric,control_mean_seconds,test_mean_seconds,difference_test_minus_control_seconds,t_statistic,p_value,alpha,result,conclusion
0,Raw,avg_iteration_time_seconds,54.50,56.66,2.16,3.5278,0.000419,0.05,Reject H0,There is a statistically significant differenc...
1,IQR,avg_iteration_time_seconds,41.82,42.06,0.24,0.8533,0.393495,0.05,Fail to reject H0,There is not enough evidence to confirm a stat...


H0: avg_duration_Test >= avg_duration_Control  

H1: avg_duration_Test < avg_duration_Control

In [77]:
# =========================
# A/B Testing - Average Iteration Duration
# Raw + IQR Welch t-test
# =========================

from scipy.stats import ttest_ind
import pandas as pd

# Hypotheses:
# H0: avg_duration_Test = avg_duration_Control
# H1: avg_duration_Test != avg_duration_Control

alpha = 0.05

# =========================
# 1. Raw duration test
# =========================

control_duration_raw = client_iteration_time.loc[
    client_iteration_time["Variation"] == "Control",
    "avg_iteration_time_seconds"
]

test_duration_raw = client_iteration_time.loc[
    client_iteration_time["Variation"] == "Test",
    "avg_iteration_time_seconds"
]

t_stat_raw, p_value_raw = ttest_ind(
    test_duration_raw,
    control_duration_raw,
    equal_var=False,
    alternative="two-sided"
)

control_mean_raw = control_duration_raw.mean()
test_mean_raw = test_duration_raw.mean()
duration_difference_raw = test_mean_raw - control_mean_raw

raw_result = "Reject H0" if p_value_raw < alpha else "Fail to reject H0"

raw_conclusion = (
    "There is a statistically significant difference in average duration between Test and Control."
    if p_value_raw < alpha
    else "There is not enough evidence to confirm a statistically significant difference in average duration between Test and Control."
)

# =========================
# 2. IQR duration test
# =========================

client_iteration_time_iqr = remove_outliers_iqr_by_group(
    client_iteration_time,
    value_col="avg_iteration_time_seconds",
    group_col="Variation",
    multiplier=1.5
)

control_duration_iqr = client_iteration_time_iqr.loc[
    client_iteration_time_iqr["Variation"] == "Control",
    "avg_iteration_time_seconds"
]

test_duration_iqr = client_iteration_time_iqr.loc[
    client_iteration_time_iqr["Variation"] == "Test",
    "avg_iteration_time_seconds"
]

t_stat_iqr, p_value_iqr = ttest_ind(
    test_duration_iqr,
    control_duration_iqr,
    equal_var=False,
    alternative="two-sided"
)

control_mean_iqr = control_duration_iqr.mean()
test_mean_iqr = test_duration_iqr.mean()
duration_difference_iqr = test_mean_iqr - control_mean_iqr

iqr_result = "Reject H0" if p_value_iqr < alpha else "Fail to reject H0"

iqr_conclusion = (
    "There is a statistically significant difference in average duration between Test and Control after IQR outlier treatment."
    if p_value_iqr < alpha
    else "There is not enough evidence to confirm a statistically significant difference in average duration between Test and Control after IQR outlier treatment."
)

# =========================
# 3. Summary table
# =========================

ab_duration_result = pd.DataFrame({
    "method": ["Raw", "IQR"],
    "metric": ["avg_iteration_time_seconds", "avg_iteration_time_seconds"],
    "control_mean_seconds": [round(control_mean_raw, 2), round(control_mean_iqr, 2)],
    "test_mean_seconds": [round(test_mean_raw, 2), round(test_mean_iqr, 2)],
    "difference_test_minus_control_seconds": [round(duration_difference_raw, 2), round(duration_difference_iqr, 2)],
    "t_statistic": [round(t_stat_raw, 4), round(t_stat_iqr, 4)],
    "p_value": [round(p_value_raw, 6), round(p_value_iqr, 6)],
    "alpha": [alpha, alpha],
    "result": [raw_result, iqr_result],
    "conclusion": [raw_conclusion, iqr_conclusion]
})

ab_duration_result

,method,metric,control_mean_seconds,test_mean_seconds,difference_test_minus_control_seconds,t_statistic,p_value,alpha,result,conclusion
0,Raw,avg_iteration_time_seconds,54.50,56.66,2.16,3.5278,0.000419,0.05,Reject H0,There is a statistically significant differenc...
1,IQR,avg_iteration_time_seconds,41.82,42.06,0.24,0.8533,0.393495,0.05,Fail to reject H0,There is not enough evidence to confirm a stat...


A Welch two-sample t-test was used to compare the average iteration duration between the Test and Control groups.

The test was performed in two ways. First, the raw data was tested without removing any observations. Second, the same test was repeated after applying IQR-based outlier treatment within each Variation group.

The raw test checks the full observed behaviour, while the IQR-adjusted test provides a more robust comparison by reducing the influence of extreme duration values.

A two-sided test was used because the objective was to evaluate whether the average duration differed between Test and Control, regardless of direction.

The raw test showed a statistically significant difference in average iteration duration, with the Test group taking slightly longer than Control.

However, after applying IQR-based outlier treatment, the difference became very small and was no longer statistically significant. This suggests that the raw duration difference was mainly driven by extreme values rather than a meaningful difference in typical user behaviour.

H0: backtracking_rate_Test >= backtracking_rate_Control

H1: backtracking_rate_Test < backtracking_rate_Control



In [79]:
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind
import pandas as pd
import numpy as np

alpha = 0.05

In [80]:
# =========================
# A/B Testing - Backtracking Rate
# =========================

# Extract counts
control_backtracked = backtracking_rate_kpi.loc[
    backtracking_rate_kpi["Variation"] == "Control",
    "clients_who_backtracked"
].iloc[0]

test_backtracked = backtracking_rate_kpi.loc[
    backtracking_rate_kpi["Variation"] == "Test",
    "clients_who_backtracked"
].iloc[0]

control_started = backtracking_rate_kpi.loc[
    backtracking_rate_kpi["Variation"] == "Control",
    "started_clients"
].iloc[0]

test_started = backtracking_rate_kpi.loc[
    backtracking_rate_kpi["Variation"] == "Test",
    "started_clients"
].iloc[0]

# Rates
control_backtracking_rate = control_backtracked / control_started
test_backtracking_rate = test_backtracked / test_started

# Two-proportion z-test
# H1: Test < Control
z_stat, p_value = proportions_ztest(
    count=[test_backtracked, control_backtracked],
    nobs=[test_started, control_started],
    alternative="smaller"
)

absolute_difference_pp = (test_backtracking_rate - control_backtracking_rate) * 100
relative_difference_pct = ((test_backtracking_rate / control_backtracking_rate) - 1) * 100

if p_value < alpha:
    result = "Reject H0"
    conclusion = "The Test group has a statistically significant lower backtracking rate."
else:
    result = "Fail to reject H0"
    conclusion = "There is not enough evidence to confirm that Test reduced backtracking rate."

ab_backtracking_result = pd.DataFrame({
    "metric": ["backtracking_rate"],
    "control_rate_%": [round(control_backtracking_rate * 100, 2)],
    "test_rate_%": [round(test_backtracking_rate * 100, 2)],
    "difference_test_minus_control_pp": [round(absolute_difference_pp, 2)],
    "relative_difference_%": [round(relative_difference_pct, 2)],
    "z_statistic": [round(z_stat, 4)],
    "p_value": [round(p_value, 6)],
    "alpha": [alpha],
    "result": [result],
    "conclusion": [conclusion]
})

ab_backtracking_result

,metric,control_rate_%,test_rate_%,difference_test_minus_control_pp,relative_difference_%,z_statistic,p_value,alpha,result,conclusion
0,backtracking_rate,26.14,33.72,7.58,28.98,18.426,1.0,0.05,Fail to reject H0,There is not enough evidence to confirm that T...



H0: error_rate_Test = error_rate_Control

H1: error_rate_Test != error_rate_Control

((started_sessions_with_errors / started_sessions))


In [81]:
# =========================
# A/B Testing - Error Rate
# =========================

# Extract counts
control_error_sessions = error_rate_kpi.loc[
    error_rate_kpi["Variation"] == "Control",
    "started_sessions_with_errors"
].iloc[0]

test_error_sessions = error_rate_kpi.loc[
    error_rate_kpi["Variation"] == "Test",
    "started_sessions_with_errors"
].iloc[0]

control_started_sessions = error_rate_kpi.loc[
    error_rate_kpi["Variation"] == "Control",
    "started_sessions"
].iloc[0]

test_started_sessions = error_rate_kpi.loc[
    error_rate_kpi["Variation"] == "Test",
    "started_sessions"
].iloc[0]

# Rates
control_error_rate = control_error_sessions / control_started_sessions
test_error_rate = test_error_sessions / test_started_sessions

# Two-proportion z-test
# H1: Test != Control
z_stat, p_value = proportions_ztest(
    count=[test_error_sessions, control_error_sessions],
    nobs=[test_started_sessions, control_started_sessions],
    alternative="two-sided"
)

absolute_difference_pp = (test_error_rate - control_error_rate) * 100
relative_difference_pct = ((test_error_rate / control_error_rate) - 1) * 100

if p_value < alpha:
    result = "Reject H0"
    conclusion = "There is a statistically significant difference in error rate between Test and Control."
else:
    result = "Fail to reject H0"
    conclusion = "There is not enough evidence to confirm a statistically significant difference in error rate."

ab_error_rate_result = pd.DataFrame({
    "metric": ["error_rate"],
    "control_rate_%": [round(control_error_rate * 100, 2)],
    "test_rate_%": [round(test_error_rate * 100, 2)],
    "difference_test_minus_control_pp": [round(absolute_difference_pp, 2)],
    "relative_difference_%": [round(relative_difference_pct, 2)],
    "z_statistic": [round(z_stat, 4)],
    "p_value": [round(p_value, 6)],
    "alpha": [alpha],
    "result": [result],
    "conclusion": [conclusion]
})

ab_error_rate_result

,metric,control_rate_%,test_rate_%,difference_test_minus_control_pp,relative_difference_%,z_statistic,p_value,alpha,result,conclusion
0,error_rate,20.77,29.79,9.02,43.43,26.2188,0.0,0.05,Reject H0,There is a statistically significant differenc...



H0: avg_step_visits_Test = avg_step_visits_Control

H1: avg_step_visits_Test != avg_step_visits_Control


In [82]:
# =========================
# A/B Testing - Average Step Visits per Client
# Raw + IQR Welch t-test
# =========================

# If client_step_visits already exists, you can skip this block
client_step_visits = (
    web_df[web_df["Variation"].isin(["Test", "Control"])]
    .groupby(["Variation", "client_id"], as_index=False)
    .agg(
        step_visits=("process_step", "count"),
        sessions=("visit_id", "nunique")
    )
)

# =========================
# 3A. Raw Welch t-test
# =========================

control_steps_raw = client_step_visits.loc[
    client_step_visits["Variation"] == "Control",
    "step_visits"
]

test_steps_raw = client_step_visits.loc[
    client_step_visits["Variation"] == "Test",
    "step_visits"
]

t_stat_raw, p_value_raw = ttest_ind(
    test_steps_raw,
    control_steps_raw,
    equal_var=False,
    alternative="two-sided"
)

control_mean_raw = control_steps_raw.mean()
test_mean_raw = test_steps_raw.mean()
difference_raw = test_mean_raw - control_mean_raw

raw_result = "Reject H0" if p_value_raw < alpha else "Fail to reject H0"

raw_conclusion = (
    "There is a statistically significant difference in average step visits between Test and Control."
    if p_value_raw < alpha
    else "There is not enough evidence to confirm a statistically significant difference in average step visits."
)

# =========================
# 3B. IQR Welch t-test
# =========================

client_step_visits_iqr = remove_outliers_iqr_by_group(
    client_step_visits,
    value_col="step_visits",
    group_col="Variation",
    multiplier=1.5
)

control_steps_iqr = client_step_visits_iqr.loc[
    client_step_visits_iqr["Variation"] == "Control",
    "step_visits"
]

test_steps_iqr = client_step_visits_iqr.loc[
    client_step_visits_iqr["Variation"] == "Test",
    "step_visits"
]

t_stat_iqr, p_value_iqr = ttest_ind(
    test_steps_iqr,
    control_steps_iqr,
    equal_var=False,
    alternative="two-sided"
)

control_mean_iqr = control_steps_iqr.mean()
test_mean_iqr = test_steps_iqr.mean()
difference_iqr = test_mean_iqr - control_mean_iqr

iqr_result = "Reject H0" if p_value_iqr < alpha else "Fail to reject H0"

iqr_conclusion = (
    "There is a statistically significant difference in average step visits between Test and Control after IQR outlier treatment."
    if p_value_iqr < alpha
    else "There is not enough evidence to confirm a statistically significant difference in average step visits after IQR outlier treatment."
)

# =========================
# 3C. Summary table
# =========================

ab_step_visits_result = pd.DataFrame({
    "method": ["Raw", "IQR"],
    "metric": ["avg_step_visits_per_client", "avg_step_visits_per_client"],
    "control_mean": [round(control_mean_raw, 2), round(control_mean_iqr, 2)],
    "test_mean": [round(test_mean_raw, 2), round(test_mean_iqr, 2)],
    "difference_test_minus_control": [round(difference_raw, 2), round(difference_iqr, 2)],
    "t_statistic": [round(t_stat_raw, 4), round(t_stat_iqr, 4)],
    "p_value": [round(p_value_raw, 6), round(p_value_iqr, 6)],
    "alpha": [alpha, alpha],
    "result": [raw_result, iqr_result],
    "conclusion": [raw_conclusion, iqr_conclusion]
})

ab_step_visits_result

,method,metric,control_mean,test_mean,difference_test_minus_control,t_statistic,p_value,alpha,result,conclusion
0,Raw,avg_step_visits_per_client,5.97,6.55,0.58,16.2480,0.0,0.05,Reject H0,There is a statistically significant differenc...
1,IQR,avg_step_visits_per_client,5.59,5.74,0.15,7.2497,0.0,0.05,Reject H0,There is a statistically significant differenc...



H0: step_completion_rate_Test <= step_completion_rate_Control

H1: step_completion_rate_Test > step_completion_rate_Control


In [83]:
# =========================
# A/B Testing - Step Completion Rate by Funnel Transition
# =========================

step_completion_ab_results = []

for _, transition in step_completion_kpi[["current_step", "next_step"]].drop_duplicates().iterrows():
    current_step = transition["current_step"]
    next_step = transition["next_step"]

    control_row = step_completion_kpi[
        (step_completion_kpi["Variation"] == "Control") &
        (step_completion_kpi["current_step"] == current_step) &
        (step_completion_kpi["next_step"] == next_step)
    ].iloc[0]

    test_row = step_completion_kpi[
        (step_completion_kpi["Variation"] == "Test") &
        (step_completion_kpi["current_step"] == current_step) &
        (step_completion_kpi["next_step"] == next_step)
    ].iloc[0]

    control_success = control_row["clients_reached_next_step"]
    test_success = test_row["clients_reached_next_step"]

    control_nobs = control_row["clients_reached_current_step"]
    test_nobs = test_row["clients_reached_current_step"]

    control_rate = control_success / control_nobs
    test_rate = test_success / test_nobs

    # H1: Test > Control
    z_stat, p_value = proportions_ztest(
        count=[test_success, control_success],
        nobs=[test_nobs, control_nobs],
        alternative="larger"
    )

    absolute_lift_pp = (test_rate - control_rate) * 100
    relative_lift_pct = ((test_rate / control_rate) - 1) * 100

    if p_value < alpha:
        result = "Reject H0"
        conclusion = "The Test group has a statistically significant higher step completion rate."
    else:
        result = "Fail to reject H0"
        conclusion = "There is not enough evidence to confirm that Test improved this step completion rate."

    step_completion_ab_results.append({
        "current_step": current_step,
        "next_step": next_step,
        "control_rate_%": round(control_rate * 100, 2),
        "test_rate_%": round(test_rate * 100, 2),
        "absolute_lift_pp": round(absolute_lift_pp, 2),
        "relative_lift_%": round(relative_lift_pct, 2),
        "z_statistic": round(z_stat, 4),
        "p_value": round(p_value, 6),
        "alpha": alpha,
        "result": result,
        "conclusion": conclusion
    })

ab_step_completion_result = pd.DataFrame(step_completion_ab_results)

ab_step_completion_result

,current_step,next_step,control_rate_%,test_rate_%,absolute_lift_pp,relative_lift_%,z_statistic,p_value,alpha,result,conclusion
0,start,step_1,86.13,90.96,4.83,5.61,17.0299,0.000000,0.05,Reject H0,The Test group has a statistically significant...
1,step_1,step_2,92.55,91.72,-0.83,-0.89,-3.2099,0.999336,0.05,Fail to reject H0,There is not enough evidence to confirm that T...
2,step_2,step_3,93.42,93.81,0.40,0.43,1.6415,0.050342,0.05,Fail to reject H0,There is not enough evidence to confirm that T...
3,step_3,confirm,88.59,89.49,0.90,1.02,2.8240,0.002372,0.05,Reject H0,The Test group has a statistically significant...


In [84]:
# =========================
# Final A/B Testing Summary - 7 Main Hypothesis Tests
# =========================

ab_testing_summary_rows = []

# 1. Completion Rate
row = ab_completion_result.iloc[0]
ab_testing_summary_rows.append({
    "test_name": "Completion Rate",
    "method": "Two-proportion z-test",
    "data_version": "Raw",
    "hypothesis": "H1: Test > Control",
    "control_value": row["control_rate_%"],
    "test_value": row["test_rate_%"],
    "difference_test_minus_control": row["absolute_lift_pp"],
    "unit": "percentage points",
    "test_statistic": row["z_statistic"],
    "p_value": row["p_value"],
    "alpha": row["alpha"],
    "result": row["result"],
    "conclusion": row["conclusion"]
})

# 2-3. Average Iteration Duration - Raw + IQR
for _, row in ab_duration_result.iterrows():
    ab_testing_summary_rows.append({
        "test_name": "Average Iteration Duration",
        "method": "Welch t-test",
        "data_version": row["method"],
        "hypothesis": "H1: Test != Control",
        "control_value": row["control_mean_seconds"],
        "test_value": row["test_mean_seconds"],
        "difference_test_minus_control": row["difference_test_minus_control_seconds"],
        "unit": "seconds",
        "test_statistic": row["t_statistic"],
        "p_value": row["p_value"],
        "alpha": row["alpha"],
        "result": row["result"],
        "conclusion": row["conclusion"]
    })

# 4. Backtracking Rate
row = ab_backtracking_result.iloc[0]
ab_testing_summary_rows.append({
    "test_name": "Backtracking Rate",
    "method": "Two-proportion z-test",
    "data_version": "Raw",
    "hypothesis": "H1: Test < Control",
    "control_value": row["control_rate_%"],
    "test_value": row["test_rate_%"],
    "difference_test_minus_control": row["difference_test_minus_control_pp"],
    "unit": "percentage points",
    "test_statistic": row["z_statistic"],
    "p_value": row["p_value"],
    "alpha": row["alpha"],
    "result": row["result"],
    "conclusion": row["conclusion"]
})

# 5. Error Rate
row = ab_error_rate_result.iloc[0]
ab_testing_summary_rows.append({
    "test_name": "Error Rate",
    "method": "Two-proportion z-test",
    "data_version": "Raw",
    "hypothesis": "H1: Test != Control",
    "control_value": row["control_rate_%"],
    "test_value": row["test_rate_%"],
    "difference_test_minus_control": row["difference_test_minus_control_pp"],
    "unit": "percentage points",
    "test_statistic": row["z_statistic"],
    "p_value": row["p_value"],
    "alpha": row["alpha"],
    "result": row["result"],
    "conclusion": row["conclusion"]
})

# 6-7. Average Step Visits per Client - Raw + IQR
for _, row in ab_step_visits_result.iterrows():
    ab_testing_summary_rows.append({
        "test_name": "Average Step Visits per Client",
        "method": "Welch t-test",
        "data_version": row["method"],
        "hypothesis": "H1: Test != Control",
        "control_value": row["control_mean"],
        "test_value": row["test_mean"],
        "difference_test_minus_control": row["difference_test_minus_control"],
        "unit": "step visits",
        "test_statistic": row["t_statistic"],
        "p_value": row["p_value"],
        "alpha": row["alpha"],
        "result": row["result"],
        "conclusion": row["conclusion"]
    })

ab_testing_summary = pd.DataFrame(ab_testing_summary_rows)

ab_testing_summary

,test_name,method,data_version,hypothesis,control_value,test_value,difference_test_minus_control,unit,test_statistic,p_value,alpha,result,conclusion
0,Completion Rate,Two-proportion z-test,Raw,H1: Test > Control,65.47,69.01,3.54,percentage points,8.4364,0.000000,0.05,Reject H0,The Test group has a statistically significant...
1,Average Iteration Duration,Welch t-test,Raw,H1: Test != Control,54.50,56.66,2.16,seconds,3.5278,0.000419,0.05,Reject H0,There is a statistically significant differenc...
2,Average Iteration Duration,Welch t-test,IQR,H1: Test != Control,41.82,42.06,0.24,seconds,0.8533,0.393495,0.05,Fail to reject H0,There is not enough evidence to confirm a stat...
3,Backtracking Rate,Two-proportion z-test,Raw,H1: Test < Control,26.14,33.72,7.58,percentage points,18.4260,1.000000,0.05,Fail to reject H0,There is not enough evidence to confirm that T...
4,Error Rate,Two-proportion z-test,Raw,H1: Test != Control,20.77,29.79,9.02,percentage points,26.2188,0.000000,0.05,Reject H0,There is a statistically significant differenc...
5,Average Step Visits per Client,Welch t-test,Raw,H1: Test != Control,5.97,6.55,0.58,step visits,16.2480,0.000000,0.05,Reject H0,There is a statistically significant differenc...
6,Average Step Visits per Client,Welch t-test,IQR,H1: Test != Control,5.59,5.74,0.15,step visits,7.2497,0.000000,0.05,Reject H0,There is a statistically significant differenc...


In [85]:
ab_testing_summary_clean = ab_testing_summary[
    [
        "test_name",
        "data_version",
        "hypothesis",
        "control_value",
        "test_value",
        "difference_test_minus_control",
        "unit",
        "p_value",
        "alpha",
        "result"
    ]
]

ab_testing_summary_clean

,test_name,data_version,hypothesis,control_value,test_value,difference_test_minus_control,unit,p_value,alpha,result
0,Completion Rate,Raw,H1: Test > Control,65.47,69.01,3.54,percentage points,0.000000,0.05,Reject H0
1,Average Iteration Duration,Raw,H1: Test != Control,54.50,56.66,2.16,seconds,0.000419,0.05,Reject H0
2,Average Iteration Duration,IQR,H1: Test != Control,41.82,42.06,0.24,seconds,0.393495,0.05,Fail to reject H0
3,Backtracking Rate,Raw,H1: Test < Control,26.14,33.72,7.58,percentage points,1.000000,0.05,Fail to reject H0
4,Error Rate,Raw,H1: Test != Control,20.77,29.79,9.02,percentage points,0.000000,0.05,Reject H0
5,Average Step Visits per Client,Raw,H1: Test != Control,5.97,6.55,0.58,step visits,0.000000,0.05,Reject H0
6,Average Step Visits per Client,IQR,H1: Test != Control,5.59,5.74,0.15,step visits,0.000000,0.05,Reject H0


In [86]:
# =========================
# A/B Testing Summary - Step Completion Rate by Funnel Transition
# =========================

ab_step_completion_summary = ab_step_completion_result[
    [
        "current_step",
        "next_step",
        "control_rate_%",
        "test_rate_%",
        "absolute_lift_pp",
        "relative_lift_%",
        "z_statistic",
        "p_value",
        "alpha",
        "result",
        "conclusion"
    ]
]

ab_step_completion_summary

,current_step,next_step,control_rate_%,test_rate_%,absolute_lift_pp,relative_lift_%,z_statistic,p_value,alpha,result,conclusion
0,start,step_1,86.13,90.96,4.83,5.61,17.0299,0.000000,0.05,Reject H0,The Test group has a statistically significant...
1,step_1,step_2,92.55,91.72,-0.83,-0.89,-3.2099,0.999336,0.05,Fail to reject H0,There is not enough evidence to confirm that T...
2,step_2,step_3,93.42,93.81,0.40,0.43,1.6415,0.050342,0.05,Fail to reject H0,There is not enough evidence to confirm that T...
3,step_3,confirm,88.59,89.49,0.90,1.02,2.8240,0.002372,0.05,Reject H0,The Test group has a statistically significant...


The A/B testing summary includes the seven main hypothesis tests used to evaluate the Test experience against the Control experience.

Completion Rate, Backtracking Rate, and Error Rate were tested using two-proportion z-tests because they compare proportions between two independent groups.

Average Iteration Duration and Average Step Visits per Client were tested using Welch t-tests because they compare numerical averages between two independent groups without assuming equal variance.

For time and step-visit metrics, both raw and IQR-adjusted versions were tested. The raw version reflects the full observed behaviour, while the IQR-adjusted version reduces the influence of extreme values.

Step Completion Rate was analysed separately because it requires one hypothesis test per funnel transition.